# Timestream Lambda Function Sample Application

This notebook demonstrates generating data, according to a schema defined by the user; deploying an AWS Lambda function to process it; and visualizing the data using Grafana.

## Step 1: Generate Data

### Imports

In [1]:
import random
from datetime import datetime, timedelta, UTC
import json
import math

### Data Generator Base Class Definition

Defines the `DataGenerator` class, a base class for generating time series data. Data scenarios are defined by subclasses, in which subclasses set `_measure_templates` and `_dimension_templates` values, which define the possible values and restrictions for record measures and dimensions.

In [2]:
class DataGenerator:
    # A list of dicts used to define the format of dimensions.
    #
    # Format:
    # "name": str. Required. The name of the dimension.
    # "value_length": int. Optional. The length of the dimension value, when it is randomly generated. If neither this nor
    #     "options" are provided, defaults to 10.
    # "random_options": [str]. Optional. An array of strings to pick at random as options for the dimension value.
    #     Has precedence over "value_length". Values will be reused.
    # "unique_options": [str]. Optional. An array of strings to pick serially for the dimension value. Each value in
    #     this array will be used once.
    #
    # Example:
    # [
    #     {
    #         "name": "device_id",
    #         "value_length": 9
    #     },
    #     {
    #         "name": "region",
    #         "random_options": ["us-east-1", "us-west-2"]
    #     }
    # ]
    _dimension_templates: list

    # A list of dicts used to define the format of measures.
    #
    # Format:
    # "name": str. Required. The name of the measure.
    # "type": str. Optional. The Timestream for LiveAnalytics data type of the measure. Valid options are "DOUBLE",
    #     "BIGINT", "BOOLEAN", and "VARCHAR". Defaults to "DOUBLE".
    # "max_variation": Optional. The maximum amount a measure value can changed over time, positively or negatively.
    #     For example, with a value of 5.0, measure values will increment by a max of 5.0 and a min of -5.0. Defaults to 1.5.
    # "max": Optional. The maximum measure value. Defaults to 100.0.
    # "min": Optional. The minimum measure value. Defaults to 0.0.
    # "random_options": Optional. A list of values the measure value can have. All elements of the list should be the same data type and
    #     match the data type specified by the "type" field. This field overrides "max_variation", "max", and "min".
    #     Values will be reused, in the same way as the dimension_templates "unique_options" field.
    #
    # Example:
    # [
    #     {
    #         "name": "temperature_celsius",
    #         "type": "DOUBLE",
    #         "max_variation": 2.0,
    #         "max": 40.0,
    #         "min": 30.0
    #     },
    #     {
    #         "name": "symptoms",
    #         "type": "VARCHAR",
    #         "random_options": ["none", "headache", "shortness of breath", "fatigue", "nausea"]
    #     }
    # ]
    _measure_templates: list

    def __init__(self):
        # All subclasses need to do is provide values for _measure_templates and dimension_templates
        self._measure_templates = []
        self._dimension_templates = []

    def generate(self, start_date: datetime, end_date: datetime, reporting_frequency: timedelta,
                num_entities: int, precision="MILLISECONDS", generate_unique_options_fallback=False) -> list:
        """
        Generates time series data.

        :param start_date: The start date to use when generating records. This cannot be older in hours
            than the memory retention period in hours value for the table.
        :param end_date: The end date to use when generating records. The maximum end date
            Timestream for LiveAnalytics allows is 15 minutes in the future.
        :param reporting_frequency: The frequency that records are generated by all entities, for example,
            every 2 seconds, every 5 hours, etc.
        :param num_entities: The number of entities that will report for each timestamp, for example,
            the number of servers or number of stocks.
        :param precision: The precision to use for record timestamps. Valid options are "MILLISECONDS",
            "SECONDS", and "MICROSECONDS".
        :param generate_unique_options_fallback: Whether to generate random strings for dimension values
            after all values in a dimension template's "unique_options" array have been used.
        """

        # Construct entities (e.g., servers, weather reporting stations, stocks, etc.)
        entities = []
        for _ in range(num_entities):
            entity = {"latest_measures": {}}
            for dimension_template in self._dimension_templates:
                dimension_value_length = 20

                if "value_length" in dimension_template:
                    dimension_value_length = dimension_template["value_length"]

                # Unique options that should not be reused. Stock symbols are
                # an example of this.
                if "unique_options" in dimension_template:
                    if len(dimension_template["unique_options"]) > 0:
                        dimension_value = dimension_template["unique_options"][0]
                        # Dimensions must be unique. Each time a choice is chosen, remove it from the list.
                        dimension_template["unique_options"].pop(0)
                    elif generate_unique_options_fallback:
                        # Generate a fallback value, since we've run out of options and the user
                        # has specified that they want more unique values generated.
                        dimension_value = self._generate_random_string(dimension_value_length)
                
                # Options that are reused. Server regions are an example of this.
                elif "random_options" in dimension_template:
                    dimension_value = random.choice(dimension_template["random_options"])

                elif "unique_options" not in dimension_template and "random_options" not in dimension_template:
                    dimension_value = self._generate_random_string(dimension_value_length)

                entity[dimension_template["name"]] = dimension_value
                    
            entities.append(entity)

        records = []
        current_date = start_date
        while current_date <= end_date:
            # Each entity has a record for a single timestamp
            for entity in entities:
                dimensions = []
                for key in entity:
                    if key != "latest_measures":
                        dimension = {
                            "Name": key,
                            "Value": entity[key],
                            "DimensionValueType": "VARCHAR" # Not configurable
                        }
                        dimensions.append(dimension)

                measures = []

                for measure_template in self._measure_templates:
                    if "name" not in measure_template:
                        raise Exception(f"Measure template was missing name: {measure_template}")
                    measure_name = measure_template["name"]

                    # Optional template fields
                    measure_value_type = "DOUBLE"
                    if "type" in measure_template:
                        measure_value_type = str(measure_template["type"]).strip().upper()
                    max_variation = 1.5
                    if "max_variation" in measure_template:
                        max_variation = measure_template["max_variation"]
                    max_value = 100.0
                    if "max" in measure_template:
                        max_value = measure_template["max"]
                    min_value = 0.0
                    if "min" in measure_template:
                        min_value = measure_template["min"]
                    varchar_length = 10
                    if "varchar_length" in measure_template:
                        varchar_length = measure_template["varchar_length"]

                    measure = {
                        "MeasureName": measure_name,
                        "MeasureValueType": measure_value_type
                    }

                    measure_value = None
                    if "random_options" in measure_template:
                        measure_value = random.choice(measure_template["random_options"])
                    else:
                        if current_date == start_date:
                            if measure_value_type == "DOUBLE":
                                measure_value = random.uniform(min_value, max_value)
                            elif measure_value_type == "VARCHAR":
                                measure_value = self._generate_random_string(varchar_length)
                            elif measure_value_type == "BIGINT":
                                measure_value = int(random.uniform(min_value, max_value))
                            elif measure_value_type == "BOOLEAN":
                                measure_value = random.choice(True, False)
                            else:
                                raise Exception("Measure value type not recognized")
                        else:
                            if measure_value_type == "DOUBLE":
                                measure_value = max(min_value, min(entity["latest_measures"][measure_name] + random.uniform(-max_variation, max_variation), max_value))
                            elif measure_value_type == "VARCHAR":
                                measure_value = self._generate_random_string(varchar_length)
                            elif measure_value_type == "BIGINT":
                                measure_value = int(max(min_value, min(entity["latest_measures"][measure_name] + int(random.uniform(-max_variation, max_variation)), max_value)))
                            elif measure_value_type == "BOOLEAN":
                                measure_value = random.choice(True, False)
                            else:
                                raise Exception("Measure value type not recognized")
                        
                    # Store the actual value in the entity for future iteration
                    entity["latest_measures"][measure_name] = measure_value

                    # Timestream requires that all data be inserted as a string
                    measure["MeasureValue"] = str(measure_value)

                    measures.append(measure)
                
                precision = precision.strip().upper()
                if precision == "SECONDS":
                    timestamp = str(int(current_date.timestamp()))
                if precision == "MICROSECONDS":
                    timestamp = str(int(current_date.timestamp() * 1_000_000))
                else:
                    # Default to millisecond precision
                    timestamp = str(int(current_date.timestamp() * 1_000))

                record = {
                    "Dimensions": dimensions,
                    "Time": timestamp,
                    "Measures": measures
                }
                records.append(record)
            current_date += reporting_frequency
        return records
    
    def _generate_random_string(self, length: int):
        """
        Generates a random alphanumeric string.

        :param length: The length of the string to generate.
        """

        letters = "ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz0123456789"
        return ''.join(random.choice(letters) for _ in range(length))
    

### Data Generator Subclass Definitions

Defines subclasses of DataGenerator that define `_measure_templates` and `_dimension_templates`.

The following subclasses are defined:
- `DevOpsDataGenerator`: Generates generic DevOps time series data for servers.
- `IoTDateGenerator`: Generates generic IoT time series data for devices.
- `StockMarketGenerator`: Generates time series data simulating stock market prices.
- `WeatherDataGenerator`: Generates time series data simulating weather reporting for different US cities.
- `GamingDataGenerator`: Generates time series data simulating player activity in a competitive online video game.
- `AirQualityDataGenerator`: Generates time series data simulating air quality in different cities around the world.
- `PatientDataGenerator`: Generates time series data simulating the status of healthcare patients.
- `EnergyDataGenerator`: Generates time series data simulating building energy usage.
- `FlightDataGenerator`: Generates time series data simulating different airline flights and the status of in-flight planes.
- `ExchangeRateDataGenerator`: Generates time series data simulating the fluctuating exchange rates of different currency pairs.
- `CustomDataGenerator`: Allows users to define their own `_measure_templates` and `_dimension_templates` to generate data of their choosing.

In [3]:
class DevOpsDataGenerator(DataGenerator):
    def __init__(self):
        self._measure_templates = [
            {
                "name": "cpu_usage",
                "type": "DOUBLE",
                "max_variation": 1.2,
                "max": 100.0,
                "min": 0.0
            },
            {
                "name": "mem_usage",
                "type": "DOUBLE",
                "max_variation": 2.3,
                "max": 100.0,
                "min": 0.0
            },
            {
                "name": "disk_usage",
                "type": "DOUBLE",
                "max_variation": 0.5,
                "max": 100.0,
                "min": 0.0
            },
            {
                "name": "network_in",
                "type": "DOUBLE",
                "max_variation": 100,
                "max": 5000,
                "min": 0
            },
            {
                "name": "network_out",
                "type": "DOUBLE",
                "max_variation": 20,
                "max": 2000,
                "min": 0
            }
        ]
        self._dimension_templates = [
            {
                "name": "server_id",
                "value_length": 14
            },
            {
                "name": "region",
                "random_options": ["ca-central-1", "ca-west-1", "us-east-1", "us-east-2", "us-west-1", "us-west-2", "sa-east-1", "eu-central-1", "eu-west-1", "eu-west-2", "eu-south-1", "eu-west-3"]
            }
        ]

class IoTDataGenerator(DataGenerator):
    def __init__(self):
        self._measure_templates = [
            {
                "name": "temperature_celsius",
                "type": "DOUBLE",
                "max_variation": 0.5,
                "max": 60,
                "min": -30
            },
            {
                "name": "relative_humidity",
                "type": "DOUBLE",
                "max_variation": 0.3,
                "max": 100.0,
                "min": 0.0
            },
            {
                "name": "battery_level",
                "type": "BIGINT",
                "max_variation": 1,
                "max": 100,
                "min": 1 # All devices have enough battery to report
            },
            {
                "name": "velocity",
                "type": "DOUBLE",
                "max_variation": 4.2,
                "max": 100.0,
                "min": 0.0
            }
        ]
        self._dimension_templates = [
            {
                "name": "device_id",
                "value_length": 14
            }
        ]

class StockMarketDataGenerator(DataGenerator):
    def __init__(self):
        self._measure_templates = [
            {
                "name": "volume",
                "type": "BIGINT",
                "max_variation": 30000,
                "max": 100000000,
                "min": 1
            },
            {
                "name": "market_cap",
                "type": "BIGINT",
                "max_variation": 100,
                "max": 100000000000,
                "min": 1000000000,
            },
            {
                "name": "price_change",
                "type": "DOUBLE",
                "max_variation": 0.5,
                "max": 1000.0,
                "min": 1.0
            },
            {
                "name": "percentage_change",
                "type": "DOUBLE",
                "max_variation": 10.0,
                "max": 100.0,
                "min": 0.0
            }
        ]
        self._dimension_templates = [
            {
                "name": "stock_symbol",
                "unique_options": ["AAPL", "TSLA", "DJIA", "SPOT", "NFLX", "MSFT", "MCD", "PG", "KO", "MMM", "IBM", "AMZN", "VZ", "JNJ", "WMT"]
            }
        ]

    def generate(self, start_date, end_date, reporting_frequency, num_entities, precision="MILLISECONDS", generate_unique_dimension_fallback=False):
        num_stock_symbols = len(self._dimension_templates[0]["unique_options"])
        if num_entities > num_stock_symbols and not generate_unique_dimension_fallback:
            raise Exception(f"num_entities ({num_entities}) was greater than the number of stock symbols ({num_stock_symbols})")
        return super().generate(start_date, end_date, reporting_frequency, num_entities, generate_unique_dimension_fallback)

class WeatherDataGenerator(DataGenerator):
    def __init__(self):
        self._measure_templates = [
            {
                "name": "temperature_celsius",
                "type": "DOUBLE",
                "max_variation": 1.5,
                "max": 65.0,
                "min": -30
            },
            {
                "name": "relative_humidity",
                "type": "DOUBLE",
                "max_variation": 0.5,
                "max": 100.0,
                "min": 0.0
            },
            {
                "name": "wind_speed_kph",
                "type": "DOUBLE",
                "max_variation": 5.5,
                "max": 407.164,
                "min": 0.0
            },
            {
                "name": "precipitation_mm",
                "type": "DOUBLE",
                "max_variation": 1.5,
                "max": 60.0,
                "min": 0.0
            },
            {
                "name": "cloud_percentage",
                "type": "DOUBLE",
                "max_variation": 10.0,
                "max": 100.0,
                "min": 0.0
            },
            {
                "name": "pressure_hpa",
                "type": "BIGINT",
                "max_variation": 10,
                "max": 1050,
                "min": 950
            },
            {
                "name": "visibility_km",
                "type": "BIGINT",
                "max_variation": 20,
                "max": 200,
                "min": 1
            }
        ]
        self._dimension_templates = [
            {
                "name": "location", 
                "unique_options": ["San Francisco, CA", "Chicago, IL", "New York, NY", "Miami, FL", "Dallas, TX", "Gary, IN", "Las Vegas, NV", "San Diego, CA", "Portland, OR", "Seattle, WA", "New Orleans, LA", "Fargo, ND", "Albuquerque, NM"]
            }
        ]

    def generate(self, start_date, end_date, reporting_frequency, num_entities, precision="MILLISECONDS", generate_unique_options_fallback=False):
        num_locations = len(self._dimension_templates[0]["unique_options"])
        if num_entities > num_locations and not generate_unique_options_fallback:
            raise Exception(f"num_entities ({num_entities}) was greater than the number of locations ({num_locations})")
        return super().generate(start_date, end_date, reporting_frequency, num_entities, generate_unique_options_fallback)

class GamingDataGenerator(DataGenerator):
    def __init__(self):
        self._measure_templates = [
            {
                "name": "X",
                "type": "DOUBLE",
                "max_variation": 2.1,
                "max": 5000.0,
                "min": -5000.0
            },
            {
                "name": "Y",
                "type": "DOUBLE",
                "max_variation": 2.1,
                "max": 5000.0,
                "min": -5000.0
            },
            {
                "name": "Z",
                "type": "DOUBLE",
                "max_variation": 2.1,
                "max": 5000.0,
                "min": -5000.0
            },
            {
                "name": "health",
                "type": "BIGINT",
                "max_variation": 60,
                "max": 100,
                "min": 1
            },
            {
                "name": "ping",
                "type": "BIGINT",
                "max_variation": 10,
                "max": 250,
                "min": 25
            },
            {
                "name": "current_equip_value",
                "type": "BIGINT",
                "max_variation": 150,
                "max": 1000000,
                "min": 10
            },
            {
                "name": "flash_duration",
                "type": "DOUBLE",
                "max_variation": 0.3,
                "max": 10.0,
                "min": 0.0
            },
            {
                "name": "pitch",
                "type": "DOUBLE",
                "max_variation": 20.0,
                "max": 90.0,
                "min": -90.0
            },
            {
                "name": "yaw",
                "type": "DOUBLE",
                "max_variation": 0.8,
                "max": 360.0,
                "min": 0.0
            }
        ]
        self._dimension_templates = [
            {
                "name": "player_id",
                "value_length": 25
            },
            {
                "name": "player_name",
                "value_length": 15
            },
            {
                "name": "clan",
                "random_options": ["mosdeff", "green_berets", "golden_ducks", "roberts", "club_z"]
            }
        ]

class AirQualityDataGenerator(DataGenerator):
    def __init__(self):
        self._measure_templates = [
            {
                "name": "PM2.5",
                "type": "DOUBLE",
                "max_variation": 1.0,
                "max": 150.0,
                "min": 15.0
            },
            {
                "name": "PM10",
                "type": "DOUBLE",
                "max_variation": 1.0,
                "max": 150.0,
                "min": 15.0
            },
            {
                "name": "CO_ppm",
                "type": "DOUBLE",
                "max_variation": 5.0,
                "max": 100.0,
                "min": 0.1
            },
            {
                "name": "NO2_ppb",
                "type": "DOUBLE",
                "max_variation": 1.0,
                "max": 300.0,
                "min": 0.5
            },
            {
                "name": "O2_percentage",
                "type": "DOUBLE",
                "max_variation": 1.0,
                "max": 25.0,
                "min": 20.8
            }
        ]
        self._dimension_templates = [
            {
                "name": "city",
                "unique_options": ["Los Angeles", "New York", "Vancouver", "Sydney", "Delhi", "Beijing", "London", "Miami", "Toronto", "Seattle", "Amsterdam"]
            }
        ]

    def generate(self, start_date, end_date, reporting_frequency, num_entities, precision="MILLISECONDS", generate_unique_options_fallback=False):
        num_cities = len(self._dimension_templates[0]["unique_options"])
        if num_entities > num_cities and not generate_unique_options_fallback:
            raise Exception(f"num_entities ({num_entities}) was greater than the number of cities ({num_cities})")
        return super().generate(start_date, end_date, reporting_frequency, num_entities, generate_unique_options_fallback)

class PatientDataGenerator(DataGenerator):
    def __init__(self):
        self._measure_templates = [
            {
                "name": "heart_rate_bpm",
                "type": "BIGINT",
                "max_variation": 5.0,
                "max": 100,
                "min": 60
            },
            {
                "name": "oxygen_saturation_percentage",
                "type": "DOUBLE",
                "max_variation": 5.0,
                "max": 100.0,
                "min": 60.0
            },
            {
                "name": "temperature_celsius",
                "type": "DOUBLE",
                "max_variation": 2.0,
                "max": 40.0,
                "min": 30.0
            }
        ]
        self._dimension_templates = [
            {
                "name": "patient_id",
                "value_length": 14
            }
        ]

class EnergyDataGenerator(DataGenerator):
    def __init__(self):
        self._measure_templates = [
            {
                "name": "energy_usage_kWh",
                "type": "BIGINT",
                "max_variation": 50,
                "max": 300,
                "min": 10
            },
            {
                "name": "occupancy",
                "type": "BIGINT",
                "max_variation": 20,
                "max": 100,
                "min": 0
            },
            {
                "name": "temperature_celsius",
                "type": "DOUBLE",
                "max_variation": 0.5,
                "max": 60,
                "min": -30
            },
            {
                "name": "relative_humidity",
                "type": "DOUBLE",
                "max_variation": 0.3,
                "max": 100.0,
                "min": 0.0
            }
        ]
        self._dimension_templates = [
            {
                "name": "building_id",
                "value_length": 14
            }
        ]

class FlightDataGenerator(DataGenerator):
    def __init__(self):
        self._measure_templates = [
            {
                "name": "fuel_level_gallons",
                "type": "BIGINT",
                "max_variation": 40,
                "max": 8000,
                "min": 1
            },
            {
                "name": "heading",
                "type": "BIGINT",
                "max_variation": 10,
                "max": 360,
                "min": 0
            },
            {
                "name": "air_temperature_celsius",
                "type": "DOUBLE",
                "max_variation": 2.0,
                "max": 40.0,
                "min": -90.0
            },
            {
                "name": "lat",
                "type": "DOUBLE",
                "max_variation": 0.5,
                "max": 90.0,
                "min": -90.0
            },
            {
                "name": "lon",
                "type": "DOUBLE",
                "max_variation": 0.5,
                "max": 180.0,
                "min": -180.0
            },
            {
                "name": "speed_knots",
                "type": "BIGINT",
                "max_variation": 10,
                "max": 250,
                "min": 200
            }
        ]
        self._dimension_templates = [
            {
                "name": "flight_id",
                "unique_options": ["FL123", "FL456", "FL890", "FL333", "FL100", "FL650", "FL256", "FL430", "FL211", "FL874"]
            },
            {
                "name": "aircraft_type",
                "random_options": ["Boeing 737", "Airbus A320"]
            }
        ]

    def generate(self, start_date, end_date, reporting_frequency, num_entities, generate_unique_options_fallback=False):
        num_flight_ids = len(self._dimension_templates[0]["unique_options"])
        if num_entities > num_flight_ids and not generate_unique_options_fallback:
            raise Exception(f"num_entities ({num_entities}) was greater than the number of flight IDs ({num_flight_ids})")
        return super().generate(start_date, end_date, reporting_frequency, num_entities, generate_unique_options_fallback)

class ExchangeRateDataGenerator(DataGenerator):
    def __init__(self):
        self._measure_templates = [
            {
                "name": "exchange_rate",
                "type": "DOUBLE",
                "max_variation": 1.0,
                "max": 90.0,
                "min": 0.62
            }
        ]
        self._dimension_templates = [
            {
                "name": "currency_pair",
                "unique_options": ["USD/EUR", "USD/CAD", "USD/GPP", "USD/CNY" "GBP/CAD", "GBP/JPY", "GBP/INR", "CHF/INR", "XAU/CNY", "UYU/CAD", "USD/XAU"]
            }
        ]

    def generate(self, start_date, end_date, reporting_frequency, num_entities, precision="MILLISECONDS", generate_unique_options_fallback=False):
        num_currency_pairs = len(self._dimension_templates[0]["unique_options"])
        if num_entities > num_currency_pairs and not generate_unique_options_fallback:
            raise Exception(f"num_entities ({num_entities}) was greater than the number of currency pairs ({num_currency_pairs})")
        return super().generate(start_date, end_date, reporting_frequency, num_entities, generate_unique_options_fallback)


class CustomDataGenerator(DataGenerator):
    def __init__(self, measure_templates: list, dimension_templates: list):
        self._measure_templates = measure_templates
        self._dimension_templates = dimension_templates

### Define Timestream for LiveAnalytics Settings

These variables are used later, by the Lambda function, when creating tables and ingesting records. These variables are defined here as they are used to confirm the desired time range for generated records is acceptable and calculate metrics to be used for cost estimation.

In [4]:
DATABASE_NAME = "sample_app_database"
TABLE_NAME = "sample_app_table"

# To be used later, by the Lambda function, to create the Timestream for LiveAnalytics table.
# If you created your table manually, update with the actual values you configured for your table.
# Default values when creating a new table in the AWS console.
MEM_STORE_RETENTION_PERIOD_IN_HOURS = 12
MAG_STORE_RETENTION_PERIOD_IN_DAYS = 3653 # 10 years

# The number of records to ingest to Timestream for LiveAnalytics at a time.
# Timestream for LiveAnalytics accepts a maximum of 100 records at a time.
BATCH_SIZE = 100
# BATCH_SIZE = 1

# The precision of the timestamp for each generated record. Valid options are "MILLISECONDS", "SECONDS", and "MICROSECONDS".
# This will also be included as a query parameter in the request sent to the Lambda function.
PRECISION = "MICROSECONDS"

### Generate Data

The data generator classes use the `generate` function to generate data. The arguments to `generate` are as follows:
- `start_date`: The start date to use when generating records. This cannot be older in hours than the memory retention period in hours value for the table.
- `end_date`: The end date to use when generating records. The maximum end date Timestream for LiveAnalytics allows is 15 minutes in the future.
- `reporting_frequency`: The frequency that records are generated by all entities, for example, every 2 seconds, every 5 hours, etc.
- `param num_entities` The number of entities that will report for each timestamp, for example, the number of servers or number of stocks.
- `param precision`: The precision to use for record timestamps. Valid options are `"MILLISECONDS"`, `"SECONDS"`, and `"MICROSECONDS"`.
- `generate_unique_options_fallback`: Whether to generate random strings for dimension values after all values in a dimension template's "unique_options" array have been used.

In [ ]:
# All timestamps default to UTC
end_date = datetime.now(UTC)
start_date = end_date - timedelta(hours=8)
reporting_frequency = timedelta(minutes=30)
num_entities = 10

if end_date > datetime.now(UTC) + timedelta(minutes=15):
    raise Exception("The end date for data generation cannot be more than 15 minutes in the future")
if start_date < datetime.now(UTC) - timedelta(hours=MEM_STORE_RETENTION_PERIOD_IN_HOURS):
    raise Exception(f"The start date for data generation cannot be more than {MEM_STORE_RETENTION_PERIOD_IN_HOURS} hours in the past")
if start_date >= end_date:
    raise Exception("The start date and end date for data generation are the same")
if (end_date - start_date) < reporting_frequency:
    raise Exception("The reporting frequency is too small for the data generation time range")

# By default, generate DevOps data, which simulates reporting from servers
sample_data = DevOpsDataGenerator().generate(start_date, end_date, reporting_frequency, num_entities, precision=PRECISION)

# Custom data
#measure_templates = [
#            {
#                "name": "exchange_rate",
#                "type": "DOUBLE",
#                "max_variation": 1.0,
#                "max": 90.0,
#                "min": 0.62
#            }
#        ]
#dimension_templates = [
#            {
#                "name": "currency_pair",
#                "value_length": 4,
#                "unique_options": ["USD/EUR", "USD/CAD", "USD/GPP", "USD/CNY" "GBP/CAD", "GBP/JPY", "GBP/INR", "CHF/INR", "XAU/CNY", "UYU/CAD"]
#            }
#        ]

#sample_data = CustomDataGenerator(measure_templates=measure_templates, dimension_templates=dimension_templates).generate(start_date, end_date, reporting_frequency, num_entities)

# Print generated data
print(json.dumps(sample_data, indent=2))

## Step 2: Calculate Cost Metrics

The following cell provides metrics that can be input into the [AWS pricing calculator](https://calculator.aws/#/) to give an estimate of costs for ingesting data to Timestream for LiveAnalytics.

The metrics are:

- Memory store writes.
    - This is calculated by determining the number of records that would be ingested within the `MEM_STORE_RETENTION_PERIOD_IN_HOURS` time frame.

Magnetic store writes are not calculated since Timestream for LiveAnalytics does not allow ingesting records with timestamps outside of the `MEM_STORE_RETENTION_PERIOD_IN_HOURS` time frame. In order for records to be stored in magnetic storage, they need to first be stored in memory then be moved to magnetic storage once enough time has passed.

These cost metrics may not be accurate, as there may be a delay between generating the data and ingesting it, causing some amounts of records to be put into magnetic store or rejected due to being too old.

In [ ]:
# The AWS pricing calculator only allows per second, per minute, per hour, per day, and per month.

num_records = len(sample_data)
time_diff = end_date - start_date

if time_diff <= timedelta(seconds=1):
    unit = "second"
    scaled_count = num_records
elif time_diff <= timedelta(minutes=1):
    unit = "minute"
    scaled_count = num_records
elif time_diff <= timedelta(hours=1):
    unit = "hour"
    scaled_count = num_records
elif time_diff <= timedelta(days=1):
    unit = "day"
    scaled_count = num_records
# 30 days in a month is standard for billing
elif time_diff <= timedelta(days=30):
    unit = "month"
    scaled_count = num_records
else:
    unit = "month"
    print(time_diff.days)
    # Round up, since the AWS pricing calculator does not accept decimal numbers
    scaled_count = math.ceil(num_records / (time_diff.days / 30))

print(f"Total records: {num_records}")
print(f"Memory store writes: {scaled_count} per {unit}")

## Step 3: Deploy AWS Lambda Function

The following code will construct and deploy a Lambda function that ingests data to Timestream.

### Imports

In [7]:
import boto3
import zipfile
import os
import json

### Generate and Deploy Lambda Function

In [ ]:
REGION_NAME='us-west-2'

# Initialize clients

iam_client = boto3.client('iam', region_name=REGION_NAME)
lambda_client = boto3.client('lambda', region_name=REGION_NAME)
sts_client = boto3.client('sts', region_name=REGION_NAME)
account_id = sts_client.get_caller_identity()['Account']

lambda_name = "TimestreamSampleLambda"

# Create IAM Role for Lambda
role_name = "TimestreamLambdaRole"
assume_role_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Principal": {"Service": "lambda.amazonaws.com"},
            "Action": "sts:AssumeRole"
        }
    ]
}

role_arn = ""

try:
    create_role_response = iam_client.create_role(
        RoleName=role_name,
        AssumeRolePolicyDocument=json.dumps(assume_role_policy),
        Description="Role for Lambda to write to Timestream"
    )
    print(f"Created IAM Role: {role_name}")
    role_arn = create_role_response['Role']['Arn']
except iam_client.exceptions.EntityAlreadyExistsException:
    print(f"IAM Role {role_name} already exists")
    try:
        role_arn = iam_client.get_role(RoleName=role_name)['Role']['Arn']
    except iam_client.exceptions.NoSuchEntityException:
        print("IAM Role could not be found")
        raise

# CloudWatch logs policy to be added to the role
cloudwatch_logs_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Action": [
                "logs:CreateLogGroup",
                "logs:CreateLogStream",
                "logs:PutLogEvents"
            ],
            "Resource": f"arn:aws:logs:{REGION_NAME}:{account_id}:log-group:/aws/lambda/{lambda_name}*"
        }
    ]
}

# Add the CloudWatch logs policy to the role
try:
    iam_client.put_role_policy(
        RoleName=role_name,
        PolicyName='CloudWatchLogsPolicy',
        PolicyDocument=json.dumps(cloudwatch_logs_policy)
    )
    print(f"Attached CloudWatch logs policy to role: {role_name}")
except Exception as e:
    print(f"Error attaching CloudWatch logs policy: {e}")

# Attach Policy to the IAM Role
policy_arn = "arn:aws:iam::aws:policy/AmazonTimestreamFullAccess"
iam_client.attach_role_policy(
    RoleName=role_name,
    PolicyArn=policy_arn
)

print(f"Attached Timestream write policy to {role_name}")

# Create Lambda function code
lambda_function_code = '''
import json
import os
import boto3
from botocore.exceptions import ClientError

REGION_NAME = os.environ['REGION_NAME']
DATABASE_NAME = os.environ['DATABASE_NAME']
TABLE_NAME = os.environ['TABLE_NAME']
BATCH_SIZE = int(os.environ['BATCH_SIZE'])
MEM_STORE_RETENTION_PERIOD_IN_HOURS = int(os.environ['MEM_STORE_RETENTION_PERIOD_IN_HOURS'])
MAG_STORE_RETENTION_PERIOD_IN_DAYS = int(os.environ['MAG_STORE_RETENTION_PERIOD_IN_DAYS'])

# Initialize the Timestream client
timestream_client = boto3.client('timestream-write', REGION_NAME)

# Define your table retention properties
RETENTION_PROPERTIES = {
    'MemoryStoreRetentionPeriodInHours': 24,  # Adjust as needed
    'MagneticStoreRetentionPeriodInDays': 365  # Adjust as needed
}

def create_timestream_database_and_table():
    """
    Create Timestream database and table if they do not exist.
    """
    try:
        # Create database if it does not exist
        timestream_client.create_database(DatabaseName=DATABASE_NAME)
        print(f"Database '{DATABASE_NAME}' created successfully.")
    except ClientError as e:
        if e.response['Error']['Code'] == 'ConflictException':
            print(f"Database '{DATABASE_NAME}' already exists.")
        else:
            raise e  # Raise if it's a different error

    try:
        # Create table if it does not exist
        timestream_client.create_table(
            DatabaseName=DATABASE_NAME,
            TableName=TABLE_NAME,
            RetentionProperties=RETENTION_PROPERTIES
        )
        print(f"Table '{TABLE_NAME}' created successfully in database '{DATABASE_NAME}'.")
    except ClientError as e:
        if e.response['Error']['Code'] == 'ConflictException':
            print(f"Table '{TABLE_NAME}' already exists in database '{DATABASE_NAME}'.")
        else:
            raise e  # Raise if it's a different error

def lambda_handler(event, context):
    """
    Lambda function to process the request and ingest records into Timestream.
    The function accepts a list of records, handles MULTI measure types, 
    and sends data in batches to Timestream.
    """

    print(event)
    query_params = event.get('queryStringParameters', {})
    precision = query_params.get('precision', 'MILLISECONDS')

    # Create the database and table if they do not exist
    create_timestream_database_and_table()

    try:
        # Extract the records from the event
        body = event.get('body', '{}')
        parsed_body = json.loads(body)
        records = parsed_body.get('records', [])
        if not records:
            return {
                "statusCode": 400,
                "body": json.dumps("No records found in the request.")
            }

        # Process records in batches
        for i in range(0, len(records), BATCH_SIZE):
            records_batch = records[i:i + 100]
            # Prepare the records for Timestream
            prepared_records = []

            for record in records_batch:
                dimensions = record.get("Dimensions", [])
                time_value = record.get("Time")
                measures = record.get("Measures", [])

                # Check if there are multiple measures, in which case we'll use MULTI
                if len(measures) > 1:
                    measure_value_type = 'MULTI'
                    multi_value_measure = {
                        'MeasureName': measures[0]['MeasureName'],  # Example MeasureName; Timestream expects just one for MULTI
                        'MeasureValues': [
                            {
                                'Name': m['MeasureName'],
                                'Value': m['MeasureValue'],
                                'Type': m['MeasureValueType']
                            }
                            for m in measures
                        ]
                    }
                    prepared_record = {
                        'Dimensions': dimensions,
                        'Time': time_value,
                        'TimeUnit': precision,
                        'MeasureName': multi_value_measure['MeasureName'],  # Set MULTI MeasureName
                        'MeasureValueType': measure_value_type,
                        'MeasureValues': multi_value_measure['MeasureValues']
                    }
                else:
                    # Handle the case where there is only one measure
                    measure = measures[0]
                    prepared_record = {
                        'Dimensions': dimensions,
                        'Time': time_value,
                        'TimeUnit': precision,
                        'MeasureName': measure['MeasureName'],
                        'MeasureValue': measure['MeasureValue'],
                        'MeasureValueType': measure['MeasureValueType']
                    }

                prepared_records.append(prepared_record)

            # Write to Timestream using the `write_records` API
            response = timestream_client.write_records(
                DatabaseName=DATABASE_NAME,
                TableName=TABLE_NAME,
                Records=prepared_records
            )
            print(f"Batch write successful for records {i} to {i + len(records_batch) - 1}: {response}")

        return {
            "statusCode": 200,
            "body": json.dumps(f"Successfully ingested {len(records)} records into Timestream.")
        }
    
    except ClientError as e:
        print(f"Failed to write to Timestream: {e}")
        return {
            "statusCode": 500,
            "body": json.dumps(f"Error writing to Timestream: {str(e)}")
        }
'''

# Save the Lambda function code to a file
lambda_function_file = "lambda_function.py"
with open(lambda_function_file, 'w') as f:
    f.write(lambda_function_code)

# Create a deployment package (zip file)
lambda_zip = "lambda_function.zip"
with zipfile.ZipFile(lambda_zip, 'w') as zipf:
    zipf.write(lambda_function_file)

try:
    with open(lambda_zip, 'rb') as f:
        lambda_client.create_function(
            FunctionName=lambda_name,
            Runtime='python3.12',
            Role=role_arn,
            Handler='lambda_function.lambda_handler',
            Architectures=['arm64'],
            Code={'ZipFile': f.read()},
            Environment={
                'Variables': {
                    'REGION_NAME': REGION_NAME,
                    'DATABASE_NAME': DATABASE_NAME,
                    'TABLE_NAME': TABLE_NAME,
                    'MEM_STORE_RETENTION_PERIOD_IN_HOURS': str(MEM_STORE_RETENTION_PERIOD_IN_HOURS),
                    'MAG_STORE_RETENTION_PERIOD_IN_DAYS': str(MAG_STORE_RETENTION_PERIOD_IN_DAYS),
                    'BATCH_SIZE': str(BATCH_SIZE)
                }
            },
            Timeout=30,
            MemorySize=128
        )
    print(f"Lambda function {lambda_name} created successfully.")
except lambda_client.exceptions.ResourceConflictException:
    print(f"Lambda function {lambda_name} already exists.")

# Clean up the files
os.remove(lambda_function_file)
os.remove(lambda_zip)

# Add a resource policy to allow invocation via the function URL

try:
    lambda_client.add_permission(
        FunctionName=lambda_name,
        StatementId='FunctionURLAllowInvoke',
        Action='lambda:InvokeFunctionUrl',
        Principal=role_arn,
        FunctionUrlAuthType='AWS_IAM'
    )
    print(f"Added resource policy to allow function URL invocation for {lambda_name}.")
except lambda_client.exceptions.ResourceConflictException:
    print(f"Resource policy for {lambda_name} already exists.")

# Create or get the Lambda Function URL
try:
    response = lambda_client.create_function_url_config(
        FunctionName=lambda_name,
        AuthType='AWS_IAM'
    )
    function_url = response['FunctionUrl']
    print(f"Lambda Function URL: {function_url}")
except lambda_client.exceptions.ResourceConflictException:
    # If the URL configuration already exists, retrieve it
    response = lambda_client.get_function_url_config(FunctionName=lambda_name)
    function_url = response['FunctionUrl']
    print(f"Lambda Function URL (existing): {function_url}")


## Step 3: Send Data to the Lambda Function

The following code will send the generated sample data to the Lambda function's URL with SigV4 authenticated requests, ensuring requests do not exceed Lambda's limit of 6 MB.

### Imports

In [9]:
import json
import requests
import boto3
from botocore.auth import SigV4Auth
from botocore.awsrequest import AWSRequest

### Send Generated Data

In [ ]:
MAX_REQUEST_SIZE = 6 * 1024 * 1024  # 6 MB in bytes

def send_data_to_lambda(data, precision="MILLISECONDS"):
    """Sends generated data to the Lambda function in chunks."""
    region = REGION_NAME
    method = "POST"

    session = boto3.Session(region_name=region)

    # Calculate the size of the entire data payload
    data_payload = json.dumps({'records': data})
    total_size = len(data_payload.encode('utf-8'))

    # Check if the total size exceeds the maximum request size
    if total_size <= MAX_REQUEST_SIZE:
        send_request(session, method, data_payload, precision)
    else:
        # Chunk the data if it's too large
        chunk_size = MAX_REQUEST_SIZE - len(b'{"records":[]}')  # Reserve space for the JSON structure
        chunks = [data[i:i + chunk_size] for i in range(0, len(data), chunk_size)]
        for chunk in chunks:
            chunk_payload = json.dumps({'records': chunk})
            send_request(session, method, chunk_payload, precision)

def send_request(session, method, payload, precision="MILLISECONDS"):
    """Sends the request to the Lambda function."""
    request = AWSRequest(
        method=method,
        url=function_url,
        params={'precision': precision},
        headers={'Content-Type': 'application/json'},
        data=payload
    )

    SigV4Auth(session.get_credentials(), 'lambda', REGION_NAME).add_auth(request)

    try:
        response = requests.request(method, function_url, params={"precision": precision}, headers=dict(request.headers), data=payload, timeout=30)
        response.raise_for_status()
        print(f'Response Status: {response.status_code}')
        print(f'Response Body: {response.content.decode("utf-8")}')
    except Exception as e:
        print(f'Error: {e}')

# Send sample data to the Lambda function
send_data_to_lambda(sample_data, PRECISION)


## Step 4: Configure Grafana

### Grafana Configuration Steps
1. Open Grafana.
2. Add a new data source with the name "timestream_sample_app_database".
3. Create a new dashboard using the provided `dashboard.json`.